# ECG specialist rendering sensitivity

Use the dedicated e4-pulse GPU kernel and the original PULSE source installation. Python 3.10 is the upstream recommendation; an existing Python 3.9 installation must be validated by the pilot.
No Unsloth/Qwen environment changes are required. PULSE consumes ECG images in this implementation.
The fixed subset is selected without regard to prior PULSE success. Each examination is tested under
legacy/calibrated rendering and zero-shot/same-ECG/full-evidence prompts. Default: 100 × 2 × 3 = 600 tasks.
All pairs use the same model, cap, prompt, and new inference run. Compare completion as well as agreement;
do not select only the 96 prior successful zero-shot cases. Changing layout may affect processor tiling.
This pilot tests the rendering confound, not clinical validation or a general specialist-model advantage.

Prompt revision v2 uses a query-only zero-shot instruction. Evidence and laboratory instructions appear only when those components are supplied. Restart the kernel before running this notebook. The original pilot and Qwen logs are preserved. All PULSE conditions run afresh, with the same model revision, inputs and budgets. Do not archive the original populated protocol.


In [ ]:
from pathlib import Path
import os, sys
CODE_ROOT = Path(os.environ.get("EXPERIMENT4_CODE_ROOT", str(Path.cwd())))
if not (CODE_ROOT / "e4_review_controls.py").is_file():
    CODE_ROOT = Path.cwd()
if not (CODE_ROOT / "e4_review_controls.py").is_file():
    raise FileNotFoundError("Set EXPERIMENT4_CODE_ROOT to the uploaded Experiment 4 code folder")
if str(CODE_ROOT) not in sys.path: sys.path.insert(0, str(CODE_ROOT))
os.environ.setdefault("EXPERIMENT4_ROOT", str(Path(os.environ.get("ECG_CXR_BRIDGE_ARTIFACT_ROOT", str(CODE_ROOT.parent / "artifacts"))) / "experiment4"))
os.environ.setdefault("HF_HOME", str(CODE_ROOT.parent / ".cache" / "huggingface"))
SOURCE_ROOT = Path(os.environ['EXPERIMENT4_ROOT'])
ORIGINAL_REVIEW_ROOT = Path(os.environ.get('E4_REVIEW_ROOT', str(SOURCE_ROOT / 'review_extension_v1')))
from e4_review_pulse_v2 import prepare
REVIEW_ROOT = prepare(ORIGINAL_REVIEW_ROOT, CODE_ROOT)
print({'original_review_preserved': str(ORIGINAL_REVIEW_ROOT), 'pulse_v2': str(REVIEW_ROOT)})


In [ ]:
from huggingface_hub import login, get_token
from getpass import getpass
token_path = Path(os.environ.get("HF_TOKEN_FILE", ""))
token = token_path.read_text().strip() if token_path.is_file() else (os.environ.get("HF_TOKEN") or get_token())
if not token: token = getpass("Hugging Face read token: ").strip()
if any(c.isspace() for c in token): raise ValueError("Token file must contain exactly one token")
login(token=token, add_to_git_credential=False)
del token
print("Hugging Face authentication configured; token not displayed.")


In [ ]:
# Change to True only after inspecting Notebook 18's images.
IMAGES_APPROVED = False
os.environ["E4_REVIEW_IMAGES_APPROVED"] = "1" if IMAGES_APPROVED else "0"
print("Python executable:", sys.executable)
print("Python version:", sys.version)
from e4_ecg_specialists import validate_specialist_environment
validate_specialist_environment("pulse7b_ecg")


In [ ]:
from e4_pulse_generation import build_pulse_prompt, PROMPT_VERSION
assert PROMPT_VERSION == 'pulse_conditional_plain_report_v2'
print('Prompt version:', PROMPT_VERSION)
print(build_pulse_prompt({'condition': 'zero_shot', 'evidence': []}))


In [ ]:
from e4_review_controls import run_generation
run_generation("pulse7b_ecg", dest=REVIEW_ROOT, max_rows=int(os.environ.get("E4_REVIEW_MAX_ROWS", "50")),
               component_cap=120, max_new_tokens=512, max_input_tokens=3584)
